<a href="https://colab.research.google.com/github/gusbrz/Removedor-de-Marca-d-agua/blob/main/marcadores_linguagem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import zipfile
import os

zip_path = '/content/workspace-01a05546-aa31-701d-87d6-d6376cd66856.zip'
extract_dir = '/content/extracted_data'

# Create the extraction directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"File extracted to: {extract_dir}")

# List the contents of the extracted directory
print("Contents of extracted directory:")
for root, dirs, files in os.walk(extract_dir):
    for name in files:
        print(os.path.join(root, name))
    for name in dirs:
        print(os.path.join(root, name))

File extracted to: /content/extracted_data
Contents of extracted directory:
/content/extracted_data/marcadores_ia
/content/extracted_data/marcadores_ia/README.md
/content/extracted_data/marcadores_ia/scripts
/content/extracted_data/marcadores_ia/exemplos_teste
/content/extracted_data/marcadores_ia/analisador
/content/extracted_data/marcadores_ia/dados
/content/extracted_data/marcadores_ia/scripts/seeds_parte1.py
/content/extracted_data/marcadores_ia/scripts/seeds_parte2.py
/content/extracted_data/marcadores_ia/scripts/criar_base.py
/content/extracted_data/marcadores_ia/exemplos_teste/social_humano.txt
/content/extracted_data/marcadores_ia/exemplos_teste/jornal_ia.txt
/content/extracted_data/marcadores_ia/exemplos_teste/jornal_humano.txt
/content/extracted_data/marcadores_ia/exemplos_teste/juridico_ia.txt
/content/extracted_data/marcadores_ia/exemplos_teste/juridico_humano.txt
/content/extracted_data/marcadores_ia/exemplos_teste/social_ia.txt
/content/extracted_data/marcadores_ia/analis

In [4]:
import sqlite3
import pandas as pd

db_path = '/content/extracted_data/marcadores_ia/dados/base_marcadores_ia.db'

try:
    # Connect to the SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Query the sqlite_master table to get the schema
    cursor.execute("SELECT name, type, sql FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    if not tables:
        print("No tables found in the database.")
    else:
        print("Database Schema:")
        for table_name, table_type, table_sql in tables:
            print(f"\nTable: {table_name} (Type: {table_type})")
            print(f"CREATE Statement: {table_sql}")

            # Optionally, display column info for each table
            cursor.execute(f"PRAGMA table_info({table_name});")
            columns = cursor.fetchall()
            if columns:
                print("Columns:")
                for col in columns:
                    print(f"  - Name: {col[1]}, Type: {col[2]}, Not Null: {bool(col[3])}, Default: {col[4]}, PK: {bool(col[5])}")

except sqlite3.Error as e:
    print(f"Error connecting to or querying the database: {e}")
finally:
    if conn:
        conn.close()


Database Schema:

Table: meta (Type: table)
CREATE Statement: CREATE TABLE meta (
    chave  TEXT PRIMARY KEY,
    valor  TEXT NOT NULL
)
Columns:
  - Name: chave, Type: TEXT, Not Null: False, Default: None, PK: True
  - Name: valor, Type: TEXT, Not Null: True, Default: None, PK: False

Table: dominios (Type: table)
CREATE Statement: CREATE TABLE dominios (
    codigo     TEXT PRIMARY KEY,
    nome       TEXT NOT NULL,
    descricao  TEXT
)
Columns:
  - Name: codigo, Type: TEXT, Not Null: False, Default: None, PK: True
  - Name: nome, Type: TEXT, Not Null: True, Default: None, PK: False
  - Name: descricao, Type: TEXT, Not Null: False, Default: None, PK: False

Table: categorias (Type: table)
CREATE Statement: CREATE TABLE categorias (
    codigo     TEXT PRIMARY KEY,
    nome       TEXT NOT NULL,
    eixo       TEXT NOT NULL,
    descricao  TEXT
)
Columns:
  - Name: codigo, Type: TEXT, Not Null: False, Default: None, PK: True
  - Name: nome, Type: TEXT, Not Null: True, Default: None, 

In [8]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 60.7 MB/s eta 0:00:00


In [9]:
import fitz
import unicodedata
import re

def extrair_e_limpar_pdf(caminho_pdf):
    """Extrai texto do PDF e realiza limpeza de caracteres e normalização."""
    doc = fitz.open(caminho_pdf)
    texto_bruto = ""
    for pagina in doc:
        texto_bruto += pagina.get_text("text") + "\n"

    # Normalização Unicode (NFKC) para tratar ligaduras e caracteres especiais
    texto_limpo = unicodedata.normalize('NFKC', texto_bruto)

    # Remoção de excesso de espaços e quebras de linha órfãs comuns em PDFs
    texto_limpo = re.sub(r'\s+', ' ', texto_limpo)

    return texto_limpo.strip()

# Lista de arquivos PDF selecionados pelo usuário
pdfs = [
    '/content/AHS-10-07-2024.pdf',
    '/content/A-HORA-DO-SUL-20250429.pdf',
    '/content/A-HORA-DO-SUL-2025-06-30.pdf',
    '/content/A-HORA-DO-SUL-2025-06-12.pdf',
    '/content/A-HORA-DO-SUL-2025-12-12.pdf'
]

# Testando a extração no primeiro PDF
texto_jornal = extrair_e_limpar_pdf(pdfs[0])
print(f"Texto extraído de {pdfs[0].split('/')[-1]} (primeiros 500 caracteres):\n")
print(texto_jornal[:500])

Texto extraído de AHS-10-07-2024.pdf (primeiros 500 caracteres):

Quarta-feira, 10 de julho de 2024 | Ano 1 – No 4 | R$ 5,00 (dia útil) | R$ 9,00 (fi m de semana) ahoradosul.com.br Zona Sul contabiliza R$ 1,9 bilhão em prejuízos na agropecuária Demandas são apresentadas por produtores e lideranças locais ao Ministério da Agricultura EFEITO DA CHUVA Soja é a principal cultura afetada, com 50 mil hectares perdidos. Em evento ontem, na Embra- pa, representantes do governo federal ouviram pedidos por apoio em pontos como renegocia- ção de dívidas, recuperação de p


In [10]:
def calcular_score_ia(texto, df_marcadores):
    """Calcula o score de probabilidade de IA baseado nos pesos dos marcadores."""
    res = analisar_texto_raw(texto, df_marcadores)
    if res.empty:
        return 0.0, res

    score_total = res['peso_total'].sum()
    return round(score_total, 2), res

def analisar_jornal_pdf(caminho_pdf, df_marcadores):
    """Fluxo completo: extração, limpeza e análise de vestígios sintéticos."""
    texto = extrair_e_limpar_pdf(caminho_pdf)
    score, detalhe = calcular_score_ia(texto, df_marcadores)

    return {
        'arquivo': caminho_pdf.split('/')[-1],
        'score': score,
        'detalhes': detalhe
    }

# Executando a análise nos PDFs carregados
print("Iniciando processamento dos jornais...")
for pdf_path in pdfs:
    try:
        resultado = analisar_jornal_pdf(pdf_path, df_marcadores)
        print(f"\n--- Arquivo: {resultado['arquivo']} ---")
        print(f"Score Sintético: {resultado['score']}")
        if not resultado['detalhes'].empty:
            display(resultado['detalhes'].head(3))
    except Exception as e:
        print(f"Erro ao processar {pdf_path}: {e}")

NameError: name 'calcular_score_ia' is not defined

In [6]:
import sqlite3
import pandas as pd

db_path = '/content/extracted_data/marcadores_ia/dados/base_marcadores_ia.db'
conn = sqlite3.connect(db_path)

# Get the list of tables
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Tables in database:")
display(tables)

# Load the 'marcadores' table (or the first available table) into a DataFrame
if not tables.empty:
    table_name = tables['name'].iloc[0]
    df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    print(f"\nContent of table: {table_name}")
    display(df.head())

conn.close()

Tables in database:


,name
0,meta
1,dominios
2,categorias
3,marcadores
4,sqlite_sequence
5,marcadores_humanos
6,estimadores
7,heuristicas
8,subgeneros
9,fontes



Content of table: meta


,chave,valor
0,nome,Base de marcadores de linguagem sintética (LLM)
1,versao,0.1.0
2,data_criacao,2026-08-31
3,idioma_principal,pt-BR (com marcadores vestígio de inglês)
4,criterio_pesos,peso bruto 0.3–4.0; multiplicadores por domíni...


In [7]:
import sqlite3
import pandas as pd
import re

# 1. Carregar a tabela de marcadores
db_path = '/content/extracted_data/marcadores_ia/dados/base_marcadores_ia.db'
conn = sqlite3.connect(db_path)
df_marcadores = pd.read_sql_query("SELECT * FROM marcadores", conn)
conn.close()

print(f"Total de marcadores carregados: {len(df_marcadores)}")
display(df_marcadores[['codigo', 'rotulo', 'padrao', 'peso']].head())

def analisar_texto_raw(texto, df_padoes):
    """Analisa o texto bruto em busca de padrões de marcadores sintéticos."""
    resultados = []
    for _, row in df_padoes.iterrows():
        padrao = row['padrao']
        try:
            # Busca o padrão (regex) no texto
            matches = re.findall(padrao, texto, re.IGNORECASE)
            if matches:
                resultados.append({
                    'codigo': row['codigo'],
                    'rotulo': row['rotulo'],
                    'ocorrencias': len(matches),
                    'peso_total': len(matches) * row['peso']
                })
        except Exception as e:
            continue

    return pd.DataFrame(resultados).sort_values(by='peso_total', ascending=False)

# Exemplo de uso (com um dos arquivos extraídos)
path_teste = '/content/extracted_data/marcadores_ia/exemplos_teste/jornal_ia.txt'
with open(path_teste, 'r', encoding='utf-8') as f:
    texto_exemplo = f.read()

print(f"\nAnálise preliminar do arquivo: {path_teste.split('/')[-1]}")
res_analise = analisar_texto_raw(texto_exemplo, df_marcadores)
display(res_analise)

Total de marcadores carregados: 129


,codigo,rotulo,padrao,peso
0,lex_cenario_atual,Fórmulas de cenário/panorama atual,\b(no|em|para o|para um|dentro do) (cen[áa]rio...,1.8
1,lex_era_digital,Na era digital/da informação,\bna era (digital|da informa[çc][ãa]o|da intel...,2.0
2,lex_mundo_cada_vez_mais,Em um mundo cada vez mais...,\bem um (mundo|cen[áa]rio|contexto|paisagem) c...,2.2
3,lex_em_um_mundo_onde,Em um mundo onde...,\b(em um|num) mundo onde\b,2.0
4,lex_constante_evolucao,Em constante evolução/mudança,\bem constante (evolu[çc][ãa]o|mudan[çc]a|tran...,1.6



Análise preliminar do arquivo: jornal_ia.txt


,codigo,rotulo,ocorrencias,peso_total
19,estr_estrutura_simetrica,Blocos com títulos em paralelo (**X**: / **Y**:),3,6.6
18,pont_negrito_markdown,Negrito markdown (**texto**),3,6.0
4,lex_importante_ressaltar,Vale/cabe/é importante ressaltar,2,3.2
1,lex_mundo_cada_vez_mais,Em um mundo cada vez mais...,1,2.2
12,lex_desbloquear_potencial,Desbloquear/libertar o potencial,1,2.2
13,lex_nao_e_sobre_e_sobre,"Não é sobre X, é sobre Y",1,2.0
6,lex_paisagem_metaforica,Paisagem metafórica (landscape),1,2.0
0,lex_cenario_atual,Fórmulas de cenário/panorama atual,1,1.8
3,lex_papel_crucial,Papel crucial/fundamental,1,1.8
2,lex_constante_evolucao,Em constante evolução/mudança,1,1.6


In [ ]:
def calcular_score_ia(texto, df_marcadores):
    """Calcula o score de probabilidade de IA baseado nos pesos dos marcadores."""
    res = analisar_texto_raw(texto, df_marcadores)
    if res.empty:
        return 0, res

    score_total = res['peso_total'].sum()
    return round(score_total, 2), res

# Comparativo entre exemplo Humano e IA
arquivos = {
    'Humano': '/content/extracted_data/marcadores_ia/exemplos_teste/jornal_humano.txt',
    'IA': '/content/extracted_data/marcadores_ia/exemplos_teste/jornal_ia.txt'
}

for tipo, path in arquivos.items():
    with open(path, 'r', encoding='utf-8') as f:
        conteudo = f.read()

    score, detalhe = calcular_score_ia(conteudo, df_marcadores)
    print(f"\n--- Análise: {tipo} ---")
    print(f"Score de Vestígios Sintéticos: {score}")
    print("Principais Marcadores Encontrados:")
    display(detalhe.head(3))
